# Flight Emissions Data

We get flight data from OpenSky. We merge that with Europa data about emissions intensity by aircraft model.


## Imports and Constants

In [1]:
import datetime as dt
import zoneinfo
import json
# import json
# from pprint import pprint
# import urllib.request 
import os
# from statistics import median, mean
import logging

from tqdm import tqdm
import polars as pl
from opensky_api import OpenSkyApi

In [2]:

# will be created if doesn't exist
data_dir = "data"

# from here: https://opensky-network.org/datasets/#metadata/aircraftDatabase
mapping_csv = os.path.join(data_dir, "raw/aircraftDatabase.csv.gz")

# from here: https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/view
# https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/at_download/file
emissions_spreadsheet = os.path.join(data_dir, "raw/1.A.3.a Aviation -Annex 1 - Master emissions calculator - 2023 - Protected - v1.5_18_09_2024.xlsx")

# downloaded from
# https://data.cso.ie/
# discovered from Wikipedia: https://en.wikipedia.org/wiki/Dublin_Airport#Statistics
passenger_numbers_sheet = os.path.join(data_dir, "popular_destinations.csv")

opensky_api_secret_file = "/home/matthew/.local/share/credentials/opensky_api_key.json"

results_dir = os.path.join(data_dir, "results")
emissions_results_path = os.path.join(results_dir, "emissions.parquet")
planes_results_path = os.path.join(results_dir, "planes.parquet")

In [3]:
# microseconds per second
us_per_s = 1e6

# Melbourne and Sydney are the same timezone
tz_name = 'Australia/Sydney' 

In [4]:
airport_info = {
    # https://en.wikipedia.org/wiki/Melbourne_Airport
    "Melbourne": {
        "IATA": "MEL",
        "ICAO": "YMML",
        "WMO": 94866,
        "location": {
            'latitude': -37.673333, 
            'longitude': 144.843333,
        }
    },
    # https://en.wikipedia.org/wiki/Sydney_Airport
    "Sydney": {
        "IATA": "SYD",
        "ICAO": "YSSY",
        "WMO": 94767,
        "location": {
            'latitude': -33.946111, 
            'longitude': 151.177222
        }
    }
}

# arbitrarily choose Sydney as the airport
# look at all flights to/from Sydney from/to anywhere
# when later filter by from/to Melbourne
airport_id_type = "ICAO"
airport_id = airport_info['Sydney'][airport_id_type]
other_airport_id = airport_info['Melbourne'][airport_id_type]


In [5]:
sydney_tz = zoneinfo.ZoneInfo('Australia/Sydney')

## Download flight data from Open Sky

The OpenSky API is documented [here](https://openskynetwork.github.io/opensky-api/python.html#opensky_api.FlightData).


In [ ]:
logger = logging.getLogger("opensky_api")
#logger.addHandler(logging.StreamHandler())
logger.setLevel(logging.DEBUG)

In [7]:
api = OpenSkyApi(client_json_path=opensky_api_secret_file)

ConnectTimeout: HTTPSConnectionPool(host='auth.opensky-network.org', port=443): Max retries exceeded with url: /auth/realms/opensky-network/protocol/openid-connect/token (Caused by ConnectTimeoutError(<HTTPSConnection(host='auth.opensky-network.org', port=443) at 0x7030e82e49e0>, 'Connection to auth.opensky-network.org timed out. (connect timeout=None)'))

In [ ]:
_date = dt.date.today() - dt.timedelta(days=100)


# must include no more than 2 days in the timespan
# time_start = dt.datetime(2025, 10, 10, 0, 0, tzinfo=sydney_tz)
# time_end = dt.datetime(2025, 10, 15, 23, 59, 59, tzinfo=sydney_tz)
# time_start = dt.datetime.combine(_date, dt.time.min, tzinfo=sydney_tz)
# time_end = dt.datetime.combine(_date, dt.time.max, tzinfo=sydney_tz)

# time_start = dt.datetime.combine(_date, dt.time.min)
# time_end = dt.datetime.combine(_date, dt.time.max)

def get_epoch(t: dt.datetime) -> int:
    return round(t.timestamp())

In [ ]:
today = dt.date.today()
_dates = [today - dt.timedelta(days=d) for d in range(95, 100)]

In [ ]:
responses = []

for _date in tqdm(_dates):
    start_epoch = get_epoch(dt.datetime.combine(_date, dt.time.min))
    end_epoch = get_epoch(dt.datetime.combine(_date, dt.time.max))
    
    responses.extend(
        api.get_departures_by_airport(
            airport=airport_id, begin=start_epoch, end=end_epoch
        )
    )
    responses.extend(
        api.get_arrivals_by_airport(
            airport=airport_id, begin=start_epoch, end=end_epoch
        )
    )

In [ ]:
flight_api_data = pl.LazyFrame([f.__dict__ for f in responses])
flight_api_data.sink_csv(os.path.join(data_dir, "api_raw.csv"))
flight_api_data.sink_parquet(os.path.join(data_dir, "api_raw.parquet"))

## Process Flight Data

You can skip the previous section and start from here to read cached data from disk.

Parse datetimes, calculate duration. (e.g. `firstSeen` actually is departure time.)
We delete in-progress flights, because that duration will be wrong.

In [43]:
flights = (
    pl.scan_parquet(os.path.join(data_dir, "api_raw.parquet"))
    .filter(pl.col("estArrivalAirport").is_not_null())
    .filter(pl.col("estArrivalAirport") != pl.col("estDepartureAirport"))
    .with_columns(
        (pl.col("firstSeen") * us_per_s).cast(
            pl.Datetime()
        ).dt.convert_time_zone("Australia/Sydney"),
        (pl.col("lastSeen") * us_per_s).cast(
            pl.Datetime()
        ).dt.convert_time_zone("Australia/Sydney"),
    )
    .with_columns(
        (pl.col("lastSeen") - pl.col("firstSeen")).alias("flightDurationActual")
    )
    .select(
        pl.col("icao24").alias("ICAO24_HEX"),
        pl.col("firstSeen").alias("DEPARTURE_TIME"),
        pl.col("lastSeen").alias("ARRIVAL_TIME"),
        pl.col("flightDurationActual").alias("FLIGHT_DURATION_ACTUAL"),
        pl.col("estDepartureAirport").alias("DEPARTURE_AIRPORT"),
        pl.col("estArrivalAirport").alias("ARRIVAL_AIRPORT"),
    )
)
flights.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str
"""7c6d8c""",2025-08-06 07:55:19 AEST,2025-08-06 08:30:13 AEST,34m 54s,"""YSSY""","""YSCB"""
"""7c6af9""",2025-08-06 07:47:16 AEST,2025-08-06 08:18:09 AEST,30m 53s,"""YSSY""","""YGDO"""
"""7c6c9c""",2025-08-06 07:47:00 AEST,2025-08-06 08:44:56 AEST,57m 56s,"""YSSY""","""YBCG"""
"""7c6de4""",2025-08-06 07:45:08 AEST,2025-08-06 08:59:52 AEST,1h 14m 44s,"""YSSY""","""YMEN"""
"""7c7bcc""",2025-08-06 07:44:17 AEST,2025-08-06 08:52:05 AEST,1h 7m 48s,"""YSSY""","""YBBN"""


In [44]:
(
    flights
    .select(
        pl.col("ARRIVAL_TIME").min().alias("ARRIVAL_MIN"),
        pl.col("ARRIVAL_TIME").max().alias("ARRIVAL_MAX"),
        pl.col("DEPARTURE_TIME").min().alias("DEPARTURE_MIN"),
        pl.col("DEPARTURE_TIME").max().alias("DEPARTURE_MAX"),
    )
    .collect()
)

ARRIVAL_MIN,ARRIVAL_MAX,DEPARTURE_MIN,DEPARTURE_MAX
"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]"
2025-08-01 08:03:31 AEST,2025-08-06 08:59:52 AEST,2025-07-31 23:29:03 AEST,2025-08-06 07:55:19 AEST


In [45]:
# filter to flights between Melbourne and Sydney
# and during the time window (API might return extra)
flights = (
    flights
    .filter(
        pl.any_horizontal(
            (pl.col("DEPARTURE_AIRPORT") == pl.lit(airport_id))
            &
            (pl.col("ARRIVAL_AIRPORT") == pl.lit(other_airport_id)),
            (pl.col("DEPARTURE_AIRPORT") == pl.lit(other_airport_id))
            &
            (pl.col("ARRIVAL_AIRPORT") == pl.lit(airport_id))
        )
    )
    # .filter(pl.col("DEPARTURE_TIME") >= time_start)
    # .filter(pl.col("ARRIVAL_TIME") >= time_end)
)
flights.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str
"""7c7a3f""",2025-08-06 07:41:51 AEST,2025-08-06 08:58:48 AEST,1h 16m 57s,"""YSSY""","""YMML"""
"""7c6d26""",2025-08-06 07:29:18 AEST,2025-08-06 08:46:22 AEST,1h 17m 4s,"""YSSY""","""YMML"""
"""7c47a6""",2025-08-06 07:16:49 AEST,2025-08-06 08:36:44 AEST,1h 19m 55s,"""YSSY""","""YMML"""
"""7c6dbb""",2025-08-06 07:16:00 AEST,2025-08-06 08:31:53 AEST,1h 15m 53s,"""YSSY""","""YMML"""
"""7c6d2e""",2025-08-06 07:11:03 AEST,2025-08-06 08:22:29 AEST,1h 11m 26s,"""YSSY""","""YMML"""


In [46]:
flights.select("DEPARTURE_AIRPORT", "ARRIVAL_AIRPORT").unique().head().collect()

DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,str
"""YMML""","""YSSY"""
"""YSSY""","""YMML"""


## Join to emissions data

We download a mapping file from OpenSky, to map icao24 IDs to flight model.
Browse the files [here](https://opensky-network.org/datasets/#metadata/aircraftDatabase).
With that page, `aircraftDatabase.csv` came from [here]("https://s3.opensky-network.org/data-samples/metadata/aircraftDatabase.csv").

Within this CSV:

* `icao24` matches the OpenSki API
* `typecode` matches what's in the emissions spreadsheet as `ICAO_24`

In [47]:
def download_file(url, path):
    dir_path = os.path.dirname(path)
    os.makedirs(dir_path, exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

In [48]:
# https://opensky-network.org/datasets/#metadata/aircraftDatabase
url = "https://s3.opensky-network.org/data-samples/metadata/aircraftDatabase.csv"
download_file(url, mapping_csv)

In [49]:
mapping_df = (
    pl.scan_csv(mapping_csv, skip_rows_after_header=1)
    .select(
        pl.col("icao24").alias("ICAO24_HEX"),
        pl.col("typecode").alias("ICAO_OTHER")
    )
)
mapping_df.head().collect()

ICAO24_HEX,ICAO_OTHER
str,str
"""aa3487""","""BE36"""
"""a4fa61""","""PA31"""
"""a7a809""","""AC90"""
"""391927""","""DR40"""
"""503c21""","""ZZZZ"""


In [50]:
flight_and_type = flights.join(mapping_df, on="ICAO24_HEX", how="left")
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str
"""7c7a3f""",2025-08-06 07:41:51 AEST,2025-08-06 08:58:48 AEST,1h 16m 57s,"""YSSY""","""YMML""","""B738"""
"""7c6d26""",2025-08-06 07:29:18 AEST,2025-08-06 08:46:22 AEST,1h 17m 4s,"""YSSY""","""YMML""","""B738"""
"""7c47a6""",2025-08-06 07:16:49 AEST,2025-08-06 08:36:44 AEST,1h 19m 55s,"""YSSY""","""YMML""",null
"""7c6dbb""",2025-08-06 07:16:00 AEST,2025-08-06 08:31:53 AEST,1h 15m 53s,"""YSSY""","""YMML""","""B738"""
"""7c6d2e""",2025-08-06 07:11:03 AEST,2025-08-06 08:22:29 AEST,1h 11m 26s,"""YSSY""","""YMML""","""B738"""


In [51]:
# from here: https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/view
url = "https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/at_download/file"
download_file(url, emissions_spreadsheet)

In [52]:
# This sheet in the file is hidden in Excel
# Extract it manually.
(
    pl.read_excel(
        source=emissions_spreadsheet,
        sheet_name="database",
        read_options={
            "skip_rows": 1,
            "header_row": True,
        },
        schema_overrides={"FORECAST  DURATION": pl.Duration()},
    )
    .write_excel(os.path.join(data_dir, "spreadsheet-extracted.xlsx"))
)

In [53]:
# load the emissions spreadsheet
emissions_lookup = (
    pl.read_excel(
        source=emissions_spreadsheet,
        sheet_name="database",
        read_options={
            "skip_rows": 1,
            "header_row": True,
        },
        schema_overrides={"FORECAST  DURATION": pl.Duration()},
    )
    .lazy()
    .filter(pl.col("ICAO_ID").is_not_null())
    .filter(pl.col("ICAO_ID") != "")
    .rename(
        {
            "ICAO_ID": "ICAO_OTHER",
            "AIRCRAFT ID": "AIRCRAFT_ID",
            # watch out, there are double spaces and trailing spaces
            # in the raw data
            "FORECAST  DURATION": "DURATION_REFERENCE",
            "FORECAST CO2 (3,15 for JET-A or 3,10 for AvGAS) ": "CO2",
            "FORECAST  NOX": "NOX",
            "FORECAST  SOX": "SOX",
            "FORECAST  H20": "H2O",
            "FORECAST  CO": "CO",
            "FORECAST  HC": "HC",
            " PM Non Volatile": "PM_NON_VOLATILE",
            "PM VOLATILE (all)": "PM_VOLATILE",
            "PM TOTAL": "PM_TOTAL",
        }
    )
)

emissions_cols = [
    "CO2",
    "NOX",
    "SOX",
    "H2O",
    "CO",
    "HC",
    "PM_NON_VOLATILE",
    "PM_VOLATILE",
    "PM_TOTAL",
]

emissions_lookup.head().collect()

ICAO_OTHER,AIRCRAFT_ID,IMPACT ACFT ID,Manufacturer,One of the models associated with this aircraft type,Description,Engine Type,Number of engines,ENGINE ID LTO,LTO or CCD,CRUISE ALT,ADES,DURATION_REFERENCE,FORECAST FUEL BURNT KG,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL
str,str,str,str,str,str,str,str,str,str,i64,i64,duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A124""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",240,200,34m 1s,6430.264058,20255.331782,168.87469,5.401423,7954.237657,8.272752,3.778494,0.22115,0.689774,0.910925
"""A140""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",300,250,40m 42s,7665.318022,24145.75177,202.240391,6.438867,9481.998763,10.541993,4.727702,0.22849,0.873995,1.102485
"""A148""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",320,500,1h 16m 56s,14398.156664,45354.193491,338.863846,12.094451,17810.519114,16.794008,8.565683,0.350165,1.822895,2.17306
"""A19N""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",340,750,1h 53m 50s,21058.965284,66335.740645,475.476091,17.689524,26049.935059,23.35394,12.527249,0.43895,2.823369,3.262319
"""A20N""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",340,1000,2h 30m 19s,27751.271402,87416.504915,609.057052,23.311065,34328.321892,29.31615,16.318223,0.548512,3.792441,4.340953


Now we do the joins.

Some ICAO_OTHER hex values have a mapping to CCD but not LTO. Use that to get the Aircraft ID. Then look up aircraft ID if there's no match based on ICAO_OTHER.

"LTO" is landing and take off (the emissions while at the airport). "CCD" is cruise, control and descent (mid-flight).
I don't know what `LTO2` is, which is sometimes missing. So we just use `LTO`.

In [54]:
# look up takeoff and landing
flight_and_type = (
    flight_and_type
    .join(
        emissions_lookup
        .select("ICAO_OTHER", "AIRCRAFT_ID")
        .unique()
        , on="ICAO_OTHER", how="left"
    )
)
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str
"""7c7a3f""",2025-08-06 07:41:51 AEST,2025-08-06 08:58:48 AEST,1h 16m 57s,"""YSSY""","""YMML""","""B738""","""A21N-A"""
"""7c6d26""",2025-08-06 07:29:18 AEST,2025-08-06 08:46:22 AEST,1h 17m 4s,"""YSSY""","""YMML""","""B738""","""A21N-A"""
"""7c47a6""",2025-08-06 07:16:49 AEST,2025-08-06 08:36:44 AEST,1h 19m 55s,"""YSSY""","""YMML""",null,null
"""7c6dbb""",2025-08-06 07:16:00 AEST,2025-08-06 08:31:53 AEST,1h 15m 53s,"""YSSY""","""YMML""","""B738""","""A21N-A"""
"""7c6d2e""",2025-08-06 07:11:03 AEST,2025-08-06 08:22:29 AEST,1h 11m 26s,"""YSSY""","""YMML""","""B738""","""A21N-A"""


In [55]:
# look up takeoff and landing emissions
flight_and_type = (
    flight_and_type
    .join(
        emissions_lookup
        .filter(pl.col("LTO or CCD") == "LTO")
        .select(["AIRCRAFT_ID"] + [pl.col(c).alias(c + "_LTO") for c in emissions_cols])
        , on="AIRCRAFT_ID", how="left"
    )
)
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c7a3f""",2025-08-06 07:41:51 AEST,2025-08-06 08:58:48 AEST,1h 16m 57s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c6d26""",2025-08-06 07:29:18 AEST,2025-08-06 08:46:22 AEST,1h 17m 4s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c47a6""",2025-08-06 07:16:49 AEST,2025-08-06 08:36:44 AEST,1h 19m 55s,"""YSSY""","""YMML""",null,null,null,null,null,null,null,null,null,null,null
"""7c6dbb""",2025-08-06 07:16:00 AEST,2025-08-06 08:31:53 AEST,1h 15m 53s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c6d2e""",2025-08-06 07:11:03 AEST,2025-08-06 08:22:29 AEST,1h 11m 26s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992


Cruise, control and descent (CCD) is harder. Most have many matches, for varying duration/distance. We need to match to two rows, and linearly interpolate in between them.


In [56]:
emissions_options = (
    flight_and_type
    .join(emissions_lookup,
          on="AIRCRAFT_ID",
          how="left"
    )
)

In [57]:
emissions_lower = (
    emissions_options
    .sort("DURATION_REFERENCE", descending=False)
    .filter(pl.col("DURATION_REFERENCE") >= pl.col("FLIGHT_DURATION_ACTUAL"))
    .group_by("AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL")
    .first()
    .select(
        ["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL", pl.col("DURATION_REFERENCE").alias("DURATION_REFERENCE_LOWER")]
        + [pl.col(c).alias(c + "_LOWER") for c in emissions_cols]
    )
)

emissions_upper = (
    emissions_options
    .sort("DURATION_REFERENCE", descending=True)
    .filter(pl.col("DURATION_REFERENCE") <= pl.col("FLIGHT_DURATION_ACTUAL"))
    .group_by("AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL")
    .first()
    .select(
        ["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL", pl.col("DURATION_REFERENCE").alias("DURATION_REFERENCE_UPPER")]
        + [pl.col(c).alias(c + "_UPPER") for c in emissions_cols]
    )
)

In [58]:
flight_emissions = (
    flight_and_type
    .join(emissions_lower, on=["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL"], how="left")
    .join(emissions_upper, on=["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL"], how="left")
    .with_columns(
        ((pl.col("FLIGHT_DURATION_ACTUAL") - pl.col("DURATION_REFERENCE_LOWER")) / (pl.col("DURATION_REFERENCE_UPPER") - pl.col("DURATION_REFERENCE_LOWER"))).alias("INTERPOLATION_FACTOR")
    )
    .with_columns(
        [
            (pl.col(c + "_LOWER") + pl.col("INTERPOLATION_FACTOR") * (pl.col(c + "_UPPER") - pl.col(c + "_LOWER"))).alias(c + "_CCD")
            for c in emissions_cols
        ]
    )
    .select(
        pl.exclude([c + app for c in emissions_cols for app in ["_UPPER", "_LOWER"]])
    )
)
flight_emissions.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO,DURATION_REFERENCE_LOWER,DURATION_REFERENCE_UPPER,INTERPOLATION_FACTOR,CO2_CCD,NOX_CCD,SOX_CCD,H2O_CCD,CO_CCD,HC_CCD,PM_NON_VOLATILE_CCD,PM_VOLATILE_CCD,PM_TOTAL_CCD
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,duration[μs],duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c7a3f""",2025-08-06 07:41:51 AEST,2025-08-06 08:58:48 AEST,1h 16m 57s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.863248,9672.468208,46.61126,2.579325,3798.363927,8.097166,0.20378,0.103483,0.250343,0.353825
"""7c6d26""",2025-08-06 07:29:18 AEST,2025-08-06 08:46:22 AEST,1h 17m 4s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.859729,9687.30937,46.672338,2.583283,3804.192027,8.102367,0.204021,0.103571,0.25077,0.354342
"""7c47a6""",2025-08-06 07:16:49 AEST,2025-08-06 08:36:44 AEST,1h 19m 55s,"""YSSY""","""YMML""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""7c6dbb""",2025-08-06 07:16:00 AEST,2025-08-06 08:31:53 AEST,1h 15m 53s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.895425,9536.777581,46.052828,2.543141,3745.078445,8.049615,0.20158,0.102676,0.246431,0.349107
"""7c6d2e""",2025-08-06 07:11:03 AEST,2025-08-06 08:22:29 AEST,1h 11m 26s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 12m 25s,39m 15s,0.029648,8976.513026,43.754327,2.393737,3525.063441,7.83893,0.192294,0.099451,0.230202,0.329653


In [59]:
# Fill in empty values with the mean of the column
cols_to_fill = [c + app for c in emissions_cols for app in ["_LTO", "_CCD"]]
flight_emissions = (
    flight_emissions
    .with_columns(
        pl.col(cols_to_fill).fill_null(pl.col(cols_to_fill).mean())
    )
)

## Sanity Check

What's the total emissions per day?

The CO2 in the raw data is in kg.

In [60]:
(
    flight_emissions
    .group_by(pl.col("DEPARTURE_TIME").dt.date().alias("DEPARTURE_DATE"))
    .agg([
        (pl.col(c + "_LTO").sum() + pl.col(c + "_CCD").sum()).alias(c)
        for c in emissions_cols
    ])
    .sort("DEPARTURE_DATE")
    .collect()
)

DEPARTURE_DATE,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-08-01,2.5010e6,15711.667789,666.942798,982152.703165,2427.955395,361.486164,24.395429,78.125129,102.520557
2025-08-02,2.1381e6,14089.891815,570.153483,839618.895758,1733.74316,363.912803,18.867072,73.214785,92.081857
2025-08-03,2.8223e6,17807.022901,752.622325,1.1083e6,2507.091923,419.347391,28.739365,91.872202,120.611567
2025-08-04,2.8945e6,18242.665262,771.863366,1.1367e6,2680.142428,427.814018,26.935353,93.154901,120.090254
2025-08-05,2.5321e6,15745.230312,675.218936,994340.302371,2495.233762,355.819772,24.3442,78.35245,102.69665
2025-08-06,379125.52913,2237.648183,101.100144,148882.005683,401.03751,44.720168,3.439202,11.185668,14.624871


So there's around 2.5 thousand tonnes of CO2 burnt per day.

Is that the right ballpark?

[Google Flights](https://www.google.com/travel/flights/search?tfs=CBwQAholEgoyMDI1LTExLTEyKABqDAgDEggvbS8wNnk1N3IHCAESA01FTEABSAFwAYIBCwj___________8BmAEC) says per-passenger emissions are around 77kg on average.

How many flights? Google Flights says 34 + 24 + 13 = 71 from Syd to Melb on 12/11/2025. Double that, it's 142 each way. 

How many seats per flight? Google Flights assumes economy seats. How do they allocate emissions between classes? Maybe 160 passengers ([Wikipedia](https://en.wikipedia.org/wiki/Airbus_A320_family)).

In [61]:
flights_per_day = 71 * 2
passengers_per_flight = 160
emissions_per_passenger = 77

emissions_per_day = emissions_per_passenger * passengers_per_flight * flights_per_day
emissions_per_day

1749440

Ok, so we're in the right ballpark.

## Airport Data

Add data about the airports to each flight

In [62]:
airport_locations = pl.DataFrame([
    {
        "AIRPORT_ID": v[airport_id_type],
        "LATITUDE": v['location']['latitude'],
        "LONGITUDE": v['location']['longitude'],
    }
    for v in airport_info.values()
])
airport_locations

AIRPORT_ID,LATITUDE,LONGITUDE
str,f64,f64
"""YMML""",-37.673333,144.843333
"""YSSY""",-33.946111,151.177222


In [63]:
flight_emissions = (
    flight_emissions
    .join(
        airport_locations
            .lazy()
            .rename({"AIRPORT_ID": "DEPARTURE_AIRPORT", 
                                   "LATITUDE": "LATITUDE_DEPARTURE",
                                   "LONGITUDE": "LONGITUDE_DEPARTURE"}),
        on="DEPARTURE_AIRPORT"
    )
    .join(
        airport_locations
            .lazy()
            .rename({"AIRPORT_ID": "ARRIVAL_AIRPORT",
                                   "LATITUDE": "LATITUDE_ARRIVAL", 
                                   "LONGITUDE": "LONGITUDE_ARRIVAL"}),
        on="ARRIVAL_AIRPORT"
    )
    # calculate the angle
    # 0 is east, 90 is north
    # Use arctan2 to handle all quadrants correctly
    .with_columns(
        (pl.col("LATITUDE_ARRIVAL") - pl.col("LATITUDE_DEPARTURE")).alias("LATITUDE_DELTA"),
        (pl.col("LONGITUDE_ARRIVAL") - pl.col("LONGITUDE_DEPARTURE")).alias("LONGITUDE_DELTA"),
    )
    .with_columns(
        pl.arctan2(pl.col("LATITUDE_DELTA"), pl.col("LONGITUDE_DELTA")).degrees().alias("ANGLE")
    )
)

In [64]:
flight_emissions.select("ARRIVAL_AIRPORT", "DEPARTURE_AIRPORT", "ANGLE").unique().head().collect()

ARRIVAL_AIRPORT,DEPARTURE_AIRPORT,ANGLE
str,str,f64
"""YSSY""","""YMML""",30.474986
"""YMML""","""YSSY""",-149.525014


## Mid-Flight Data

We generate a list of timestamps which we care about.

OpenSky doesn't offer mid-flight positions for historcal flights, so we'll just interpolate.

In [65]:
# choose the second date
# (first might be partial day because of timezone boundaries)
_date = (
    flight_emissions
    .select(pl.col("DEPARTURE_TIME").dt.date().alias("DATE"))
    .sort("DATE")
    .head(2)
    .tail(1)
    .collect()
    .item()
)

In [76]:
num_slices = 24 * 60

times_lf = pl.LazyFrame({
    "TIME": pl.datetime_range(
        start=dt.datetime.combine(_date, dt.time.min, tzinfo=sydney_tz),
        end=dt.datetime.combine(_date, dt.time.max, tzinfo=sydney_tz),
        interval=dt.timedelta(days=1) / (num_slices - 1),
        eager=True
    )
})
    
times_lf.collect()

TIME
"datetime[μs, Australia/Sydney]"
2025-08-01 00:00:00 AEST
2025-08-01 00:01:00.041696 AEST
2025-08-01 00:02:00.083392 AEST
2025-08-01 00:03:00.125088 AEST
2025-08-01 00:04:00.166784 AEST
…
2025-08-01 23:54:59.792064 AEST
2025-08-01 23:55:59.833760 AEST
2025-08-01 23:56:59.875456 AEST


In [77]:
animation_data = (
    times_lf
    .join(flight_emissions, how="cross")
    .with_columns(
        (pl.col("TIME") >= pl.col("DEPARTURE_TIME")).alias("HAS_DEPARTED"),
        (pl.col("TIME") >= pl.col("ARRIVAL_TIME")).alias("HAS_ARRIVED"),
        pl.col("TIME").is_between("DEPARTURE_TIME", "ARRIVAL_TIME").alias("IN_AIR"),
        ((pl.col("TIME") - pl.col("DEPARTURE_TIME")) / (pl.col("ARRIVAL_TIME") - pl.col("DEPARTURE_TIME")))
        .clip(lower_bound=0, upper_bound=2)
        .alias("FLIGHT_PROGRESS")
    )
    .with_columns([
        (
            pl.col("FLIGHT_PROGRESS") * pl.col(c + "_CCD")
            + pl.col("HAS_DEPARTED") * pl.col(c + "_LTO")
        ).alias(c)
        for c in emissions_cols
    ])
    .with_columns(
        (pl.col("LONGITUDE_DEPARTURE") + pl.col("FLIGHT_PROGRESS") * (pl.col("LONGITUDE_ARRIVAL") - pl.col("LONGITUDE_DEPARTURE"))).alias("LONGITUDE"),
        (pl.col("LATITUDE_DEPARTURE") + pl.col("FLIGHT_PROGRESS") * (pl.col("LATITUDE_ARRIVAL") - pl.col("LATITUDE_DEPARTURE"))).alias("LATITUDE"),
    )
    .sort("TIME", "DEPARTURE_TIME", "ICAO24_HEX")
)
animation_data.collect()

TIME,ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO,DURATION_REFERENCE_LOWER,DURATION_REFERENCE_UPPER,INTERPOLATION_FACTOR,CO2_CCD,NOX_CCD,SOX_CCD,H2O_CCD,CO_CCD,HC_CCD,PM_NON_VOLATILE_CCD,PM_VOLATILE_CCD,PM_TOTAL_CCD,LATITUDE_DEPARTURE,LONGITUDE_DEPARTURE,LATITUDE_ARRIVAL,LONGITUDE_ARRIVAL,LATITUDE_DELTA,LONGITUDE_DELTA,ANGLE,HAS_DEPARTED,HAS_ARRIVED,IN_AIR,FLIGHT_PROGRESS,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL,LONGITUDE,LATITUDE
"datetime[μs, Australia/Sydney]",str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,duration[μs],duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-08-01 00:00:00 AEST,"""7c6b13""",2025-08-01 06:46:44 AEST,2025-08-01 08:07:49 AEST,1h 21m 5s,"""YMML""","""YSSY""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 53m 50s,1h 16m 56s,0.887534,47713.906789,354.228096,12.723707,18737.174295,17.531779,9.011225,0.36015,1.935414,2.295565,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c6dc9""",2025-08-01 06:53:31 AEST,2025-08-01 08:18:28 AEST,1h 24m 57s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.621921,10690.147908,50.799499,2.850706,4198.005043,8.453803,0.220278,0.109532,0.279682,0.389214,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c6b0f""",2025-08-01 07:13:05 AEST,2025-08-01 08:32:56 AEST,1h 19m 51s,"""YMML""","""YSSY""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 53m 50s,1h 16m 56s,0.920958,47012.626532,349.662013,12.536699,18461.782795,17.312522,8.878815,0.357183,1.901975,2.259158,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c6d30""",2025-08-01 07:19:39 AEST,2025-08-01 08:42:44 AEST,1h 23m 5s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.67823,10452.689311,49.822243,2.787384,4104.755449,8.370588,0.216428,0.10812,0.272836,0.380957,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c7803""",2025-08-01 07:42:40 AEST,2025-08-01 08:58:45 AEST,1h 16m 5s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.889392,9562.219574,46.157534,2.549926,3755.069473,8.058531,0.201993,0.102828,0.247164,0.349992,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-01 23:58:59.958848 AEST,"""7c6d2e""",2025-08-06 07:11:03 AEST,2025-08-06 08:22:29 AEST,1h 11m 26s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 12m 25s,39m 15s,0.029648,8976.513026,43.754327,2.393737,3525.063441,7.83893,0.192294,0.099451,0.230202,0.329653,-33.946111,151.177222,-37.673333,144.843333,-3.727222,-6.333889,-149.525014,false,false,false,0.0,0.0,0.0,0.0,0.

`animation_data` has one row per (flight, time). The emissions values are cumulative, within each flight. So for total emissions, group by time, sum across flights.

In [78]:

os.makedirs(results_dir, exist_ok=True)

In [81]:
animation_data.filter(pl.col("HAS_DEPARTED"))

TIME,ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO,DURATION_REFERENCE_LOWER,DURATION_REFERENCE_UPPER,INTERPOLATION_FACTOR,CO2_CCD,NOX_CCD,SOX_CCD,H2O_CCD,CO_CCD,HC_CCD,PM_NON_VOLATILE_CCD,PM_VOLATILE_CCD,PM_TOTAL_CCD,LATITUDE_DEPARTURE,LONGITUDE_DEPARTURE,LATITUDE_ARRIVAL,LONGITUDE_ARRIVAL,LATITUDE_DELTA,LONGITUDE_DELTA,ANGLE,HAS_DEPARTED,HAS_ARRIVED,IN_AIR,FLIGHT_PROGRESS,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL,LONGITUDE,LATITUDE
"datetime[μs, Australia/Sydney]",str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,duration[μs],duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-08-01 00:00:00 AEST,"""7c6b13""",2025-08-01 06:46:44 AEST,2025-08-01 08:07:49 AEST,1h 21m 5s,"""YMML""","""YSSY""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 53m 50s,1h 16m 56s,0.887534,47713.906789,354.228096,12.723707,18737.174295,17.531779,9.011225,0.36015,1.935414,2.295565,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c6dc9""",2025-08-01 06:53:31 AEST,2025-08-01 08:18:28 AEST,1h 24m 57s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.621921,10690.147908,50.799499,2.850706,4198.005043,8.453803,0.220278,0.109532,0.279682,0.389214,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c6b0f""",2025-08-01 07:13:05 AEST,2025-08-01 08:32:56 AEST,1h 19m 51s,"""YMML""","""YSSY""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 53m 50s,1h 16m 56s,0.920958,47012.626532,349.662013,12.536699,18461.782795,17.312522,8.878815,0.357183,1.901975,2.259158,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c6d30""",2025-08-01 07:19:39 AEST,2025-08-01 08:42:44 AEST,1h 23m 5s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.67823,10452.689311,49.822243,2.787384,4104.755449,8.370588,0.216428,0.10812,0.272836,0.380957,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333
2025-08-01 00:00:00 AEST,"""7c7803""",2025-08-01 07:42:40 AEST,2025-08-01 08:58:45 AEST,1h 16m 5s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.889392,9562.219574,46.157534,2.549926,3755.069473,8.058531,0.201993,0.102828,0.247164,0.349992,-37.673333,144.843333,-33.946111,151.177222,3.727222,6.333889,30.474986,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.843333,-37.673333


In [84]:
(
    animation_data
    .group_by("TIME")
    .agg([
        pl.col(c).sum()
        for c in emissions_cols
    ] + [
        pl.struct("ICAO24_HEX", "DEPARTURE_TIME", "ARRIVAL_TIME").filter("HAS_DEPARTED").n_unique().alias("NUM_FLIGHTS")
    ])
    .sort("TIME")
    .sink_parquet(emissions_results_path)
)
animation_emissions_data = pl.read_parquet(emissions_results_path)
animation_emissions_data

TIME,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL,NUM_FLIGHTS
"datetime[μs, Australia/Sydney]",f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
2025-08-01 00:00:00 AEST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-08-01 00:01:00.041696 AEST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-08-01 00:02:00.083392 AEST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-08-01 00:03:00.125088 AEST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2025-08-01 00:04:00.166784 AEST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
…,…,…,…,…,…,…,…,…,…,…
2025-08-01 23:54:59.792064 AEST,4.4684e6,28536.425102,1191.563096,1.7547e6,3560.301944,604.267003,42.560836,145.33833,187.899166,118
2025-08-01 23:55:59.833760 AEST,4.4691e6,28541.609431,1191.754383,1.7550e6,3560.648865,604.382431,42.567178,145.364641,187.931819,118
2025-08-01 23:56:59.875456 AEST,4.4698e6,28546.79376,1191.94567,1.7553e6,3560.995787,604.49786,42.57352,145.390952,187.964472,118


In [85]:
(
    animation_data
    .sort("ICAO24_HEX", "DEPARTURE_TIME", "TIME")
    .select("TIME", "LATITUDE", "LONGITUDE", "ANGLE", "IN_AIR")
    .sink_parquet(planes_results_path)
)
animation_plane_data = pl.read_parquet(planes_results_path)
animation_plane_data

TIME,LATITUDE,LONGITUDE,ANGLE,IN_AIR
"datetime[μs, Australia/Sydney]",f64,f64,f64,bool
2025-08-01 00:00:00 AEST,-33.946111,151.177222,-149.525014,false
2025-08-01 00:01:00.041696 AEST,-33.946111,151.177222,-149.525014,false
2025-08-01 00:02:00.083392 AEST,-33.946111,151.177222,-149.525014,false
2025-08-01 00:03:00.125088 AEST,-33.946111,151.177222,-149.525014,false
2025-08-01 00:04:00.166784 AEST,-33.946111,151.177222,-149.525014,false
…,…,…,…,…
2025-08-01 23:54:59.792064 AEST,-33.946111,151.177222,-149.525014,false
2025-08-01 23:55:59.833760 AEST,-33.946111,151.177222,-149.525014,false
2025-08-01 23:56:59.875456 AEST,-33.946111,151.177222,-149.525014,false
